## <a href="https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156165"><b>Utilizando LCEL para criar um roteiro de viagens</b></a><br/>

In [1]:
from langchain.prompts import ChatPromptTemplate # Uma interação de múltiplos prompts é um chat, por isso, vamos importar um ChatPromptTemplate

In [2]:
%pip install -qr requirements.txt

Note: you may need to restart the kernel to use updated packages.


#### <b>PASSO 1 - IMPORTS e CRIAÇÃO DA LLM</b>

In [3]:
from langchain_openai import ChatOpenAI
from os import getenv
from dotenv import load_dotenv # CARREGA A VARIÁVEL DE AMBIENTE OPENAI_KEY LIDA DO ARQUIVO .env

load_dotenv() # CARREGANDO O ARQUIVO COM A OPENAI_KEY

llm = ChatOpenAI( # INSTANCIANDO A LLM
                    model="gpt-4.1-mini",
                    temperature=0.5,
                    # 1 - OBTENDO A API KEY POR MEIO DA VARIÁVEL DE AMBIENTE OPENAI_KEY. QUE VAI FICAR ARMAZENADA NO ARQUIVO .env.
                    # 2 - AINDA É NECESSÁRIO CARREGAR ESSE ARQUIVO. VER NA PRIMEIRA CÉLULA DO NOTEBOOK
                    api_key=getenv("OPENAI_KEY")                    
                )

#### <b>PASSO 2 - CRIANDO O <i>PROMPT TEMPLATE</i> E ASSOCIANDO UM PARSER A ELE</b></br> 

<b><ol><li>CRIANDO OS MODELOS</li></ol></b>

<ul><ul><li><b>CRIANDO O PARSER E ASSOCIANDO ELE AO MODELO DE CIDADE</b></li></ul></ul>

In [4]:
from pydantic import Field,BaseModel # pydantic -> Biblioteca para validação de dados. Garante que os dados recebidos ou manipulados estejam no formato correto,
                                     # BaseModel -> Os modelos pydantic são classes que herdam BaseModel (https://docs-pydantic-dev.translate.goog/latest/concepts/models/?_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc)
                                     # Modelos possuem campos como atributos.
                                     
                                     # Field -> Para personalizar os campos do modelo (https://docs-pydantic-dev.translate.goog/latest/concepts/fields/?_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc)

class Destino(BaseModel): # A nossa classe vai estender a classe BaseModel, que terá dois campos, a cidade e o motivo de visitá-la
    cidade: str = Field(description="cidade a visitar") # Descrição do campo. Apenas informativo
    motivo: str = Field(description="motivo pelo qual é interessante visitar") # Descrição do campo. Apenas informativo  

from langchain import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser # EXISTEM DIVERSOS OUTPUT PARSERS (https://python.langchain.com/docs/concepts/output_parsers/)

parseador = JsonOutputParser(pydantic_object=Destino) # DOCUMENTAÇÃO JsonOutputParser (https://python.langchain.com/docs/how_to/output_parser_json)

template_cidade = PromptTemplate(
                                    template="""Sugira uma cidade, dado o meu interesse por {interesse}.
                                    {formatacao_de_saida_da_ia}""", # AQUI TEMOS UMA VARÍAVEL PARCIAL. UTLIZAÇÃO DA TÉCNICA DE SHOTS PARA PROMPTS
                                    input_variables=["interesse"],
                                    # A VARÍAVEL PARCIAL É UM DICIONÁRIO. FUNCIONA COMO O SHOT
                                    partial_variables={"formatacao_de_saida_da_ia":parseador.get_format_instructions()}, # PASSANDO O PARSEADOR PARA A VARIÁVEL FORMATAÇÃO DE SAÍDA.
                                                                                                   
                                ) # INSTANCIANDO PromptTemplate e INICIANDO A PARTIR DE UM TEMPLATE

<ul><ul><li><b>CRIANDO O MODELO PARA RESTAURANTES</b></li></ul></ul>

In [5]:
template_restaurante = ChatPromptTemplate.from_template("Sugira restaurantes populares entre locais na {cidade}") # INSTANCIANDO ChatPromptTemplate e INICIANDO A PARTIR DE UM TEMPLATE

<ul><ul><li><b>CRIANDO O MODELO CULTURAL</b></li></ul></ul>

In [6]:
template_cultural = ChatPromptTemplate.from_template("Sugira atividades e locais culturais em {cidade}") # INSTANCIANDO ChatPromptTemplate e INICIANDO A PARTIR DE UM TEMPLATE

<ul><ul><li><b>USANDO A <a ref="https://python.langchain.com/docs/concepts/lcel/#composition-syntax">LCEL</a></b></li></ul></ul>
<ul><ul><ul>O resultado de um vai jogando no outro</ul></ul></ul>

In [7]:
chain1 = template_cidade |llm |parseador

In [8]:
# FAZENDO UMA CHAMADA PARA O TEMPLATE
resposta = chain1.invoke({"interesse":"praias"}) # AQUI ESTAMOS CHAMANDO A PRIMEIRA CHAIN, PASSANDO O INTERESSE COMO ENTRADA
print(resposta)

{'cidade': 'Florianópolis', 'motivo': 'Possui belas praias com águas claras e é ideal para atividades ao ar livre como surf, trilhas e passeios de barco.'}


<ul><ul><ul><b>Jogando a cidade da cadeia 1 para a cadeia 2</b></ul></ul></ul>

<font style="color:lightgreen">[llm/start]</font> [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
  <ul>"prompts": [</ul>
   <ul><ul>"Human: Sugira uma cidade, dado o meu interesse por praias.\n                                    The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {\"properties\": {\"foo\": {\"title\": \"Foo\", \"description\": \"a list of strings\", \"type\": \"array\", \"items\": {\"type\": \"string\"}}}, \"required\": [\"foo\"]}\nthe object {\"foo\": [\"bar\", \"baz\"]} is a well-formatted instance of the schema. The object {\"properties\": {\"foo\": [\"bar\", \"baz\"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{\"properties\": {\"cidade\": {\"description\": \"cidade a visitar\", \"title\": \"Cidade\", \"type\": \"string\"}, \"motivo\": {\"description\": \"motivo pelo qual é interessante visitar\", \"title\": \"Motivo\", \"type\": \"string\"}}, \"required\": [\"cidade\", \"motivo\"]}\n```"</ul></ul>
  <ul>]</ul>
}<br/>
<font style="color:lightblue">[llm/end]</font> [chain:RunnableSequence > llm:ChatOpenAI] [1.53s] Exiting LLM run with output:<br/>
{
  <ul>"text": "{\n  \"cidade\": \"Florianópolis\",\n  \"motivo\": \"Florianópolis é conhecida por suas belas praias, com opções para todos os gostos, desde praias calmas para relaxar até praias com boas ondas para surf.\"\n}"</ul>
}

<font style="color:lightgreen">[llm/start]</font>[chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
<ul> "prompts": [</ul>
    <ul><ul>"Human: Sugira restaurantes populares entre locais na Florianópolis"</ul></ul>
  ]
}<br/>
<font style="color:lightblue">[llm/end]</font> [chain:RunnableSequence > llm:ChatOpenAI] [8.17s] Exiting LLM run with output:<br/>
{
  <ul>"text": "Claro! Florianópolis é conhecida por sua gastronomia diversificada, especialmente frutos do mar frescos e pratos típicos da culinária catarinense. Aqui estão alguns restaurantes populares entre os locais que você pode gostar:\n\n1. **Ostradamus** – Localizado na Lagoa da Conceição, é famoso por suas ostras frescas e pratos à base de frutos do mar. Ambiente descontraído e ótimo para quem quer experimentar a culinária local.\n\n2. **Bar do Arante** – Um clássico em Florianópolis, na Praia do Matadeiro. Conhecido pelo peixe na telha e ambiente simples, é muito frequentado por moradores e turistas que buscam comida caseira e saborosa.\n\n3. **Restaurante do Chico** – Também na Lagoa da Conceição, oferece pratos típicos da região, como sequência de camarão e mariscos, em um ambiente rústico e acolhedor.\n\n4. **Box 32** – Localizado no Mercado Público de Florianópolis, é um ponto tradicional para comer frutos do mar frescos, como a famosa sequência de camarão. Muito frequentado por locais para um almoço rápido e saboroso.\n\n5. **Canto dos Açores** – Restaurante que valoriza a culinária açoriana, muito presente na cultura local. Fica no bairro Ribeirão da Ilha, onde também é possível encontrar ostras cultivadas na região.\n\n6. **Ponto G** – Em Santo Antônio de Lisboa, é conhecido pela comida regional com um toque contemporâneo, além da vista charmosa para o mar.\n\nSe quiser, posso ajudar com sugestões específicas de pratos ou bairros para focar!"</ul>
}

In [9]:
from langchain_core.output_parsers import StrOutputParser
#from langchain.globals import set_debug
#set_debug(True)

chain2 = template_restaurante | llm |StrOutputParser() # A saída do llm é uma string, por isso, utilizamos o StrOutputParser
chain = chain1 | chain2 # PARA PODER PASSAR A CIDADE PARA O RESTAURANTE, PRECISAMOS CRIAR UMA NOVA CHAIN QUE RECEBA A PRIMEIRA E A SEGUNDA
resposta = chain.invoke({"interesse":"praias"}) 
print(resposta) 

Claro! Florianópolis é conhecida por sua excelente gastronomia, especialmente frutos do mar, mas também oferece ótimas opções de cozinha regional e contemporânea. Aqui estão alguns restaurantes populares entre os locais:

1. **Ostradamus**  
   Localizado em Ribeirão da Ilha, é famoso pelas ostras frescas e frutos do mar. Um clássico para quem quer experimentar a culinária local autêntica.

2. **Box 32**  
   Na Lagoa da Conceição, é um dos pontos mais conhecidos para pratos com frutos do mar, especialmente o camarão na moranga e a sequência de camarão.

3. **Restaurante Rancho Açoriano**  
   Também na Lagoa da Conceição, oferece pratos típicos da culinária açoriana e frutos do mar em um ambiente rústico e acolhedor.

4. **Bar do Arante**  
   Na Praia do Pântano do Sul, é famoso pelo ambiente simples e comida caseira, com destaque para o peixe frito e frutos do mar.

5. **Ponta das Caranhas**  
   Restaurante tradicional na região sul da ilha, conhecido pela qualidade dos peixes e fr

#### <b>PASSO 4 - INVOCANDO A CADEIA GERAL</b>

<ul><ul><ul><b>Jogando da cadeia 2 para a cadeia 3</b></ul></ul></ul>

<font style="color:lightgreen">[llm/start]</font>[chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
  <ul>"prompts": [
    "Human: Sugira atividades e locais culturais em Claro! Florianópolis é conhecida por sua culinária diversificada, especialmente frutos do mar frescos. Aqui estão alguns restaurantes populares entre os locais que você pode gostar de visitar:\n\n1. **Ostradamus**  \n   Localização: Ribeirão da Ilha  \n   Destaque: Especializado em ostras e frutos do mar frescos, é um dos restaurantes mais tradicionais da ilha. Ambiente agradável e comida deliciosa.\n\n2. **Box 32**  \n   Localização: Mercado Público de Florianópolis  \n   Destaque: Comida típica catarinense e frutos do mar. É um dos pontos mais procurados para experimentar pratos locais em um ambiente descontraído.\n\n3. **Restaurante Rancho Açoriano**  \n   Localização: Lagoa da Conceição  \n   Destaque: Pratos típicos da culinária açoriana e frutos do mar, com um ambiente rústico e acolhedor.\n\n4. **Bar do Deca**  \n   Localização: Lagoa da Conceição  \n   Destaque: Petiscos e pratos com frutos do mar, ótimo para um happy hour com os locais.\n\n5. **Ponta das Caranhas**  \n   Localização: Ponta das Caranhas  \n   Destaque: Vista linda para o mar e pratos de frutos do mar frescos, muito frequentado por moradores.\n\n6. **Restaurante do Chico**  \n   Localização: Campeche  \n   Destaque: Ambiente simples e comida caseira, com destaque para peixes e frutos do mar.\n\nSe quiser opções mais específicas, como comida vegetariana ou internacional, posso ajudar também!"</ul>
  <ul>]</ul>
}<br/>
<font style="color:lightblue">[llm/end]</font>[chain:RunnableSequence > llm:ChatOpenAI] [9.75s] Exiting LLM run with output:<br/>
{
  <ul>"text": "Claro! Além das ótimas opções gastronômicas que você mencionou, Florianópolis oferece uma rica programação cultural que pode complementar sua experiência na cidade. Aqui vão algumas sugestões de atividades e locais culturais para visitar:\n\n### Atividades e locais culturais em Florianópolis\n\n1. **Centro Histórico de Florianópolis**  \n   Explore as ruas do centro histórico, onde você encontra construções coloniais, igrejas antigas como a Catedral Metropolitana, e museus como o Museu Histórico de Santa Catarina (Palácio Cruz e Sousa). É um ótimo lugar para entender a história da cidade e apreciar a arquitetura açoriana.\n\n2. **Mercado Público de Florianópolis**  \n   Além de ser um ponto gastronômico famoso, o Mercado Público é um espaço cultural vibrante com feiras de artesanato, apresentações musicais e eventos culturais frequentes. Vale a pena visitar para sentir a energia local.\n\n3. **Projeto Tamar – Praia da Joaquina**  \n   Para quem gosta de natureza e conservação, o Projeto Tamar tem uma base em Florianópolis onde é possível aprender sobre a preservação das tartarugas marinhas, um símbolo importante da região.\n\n4. **Museu da Escola Catarinense**  \n   Localizado no centro, esse museu oferece exposições sobre a história da educação no estado de Santa Catarina, com acervos que remontam ao século XIX.\n\n5. **Teatro Álvaro de Carvalho (TAC)**  \n   Um dos principais espaços culturais da cidade, o TAC oferece uma programação variada com peças de teatro, shows musicais, dança e eventos culturais. Vale a pena conferir a agenda durante sua visita.\n\n6. **Feira de Artesanato da Lagoa da Conceição**  \n   A feira acontece aos finais de semana e é um ótimo lugar para comprar artesanato local, conhecer artistas e experimentar comidas típicas em um ambiente descontraído.\n\n7. **Fortalezas da Ilha**  \n   Para quem gosta de história militar, visitar as fortalezas como a Fortaleza de São José da Ponta Grossa é uma ótima pedida. Elas oferecem uma vista panorâmica e contam a história da defesa da ilha.\n\n8. **Casa da Alfândega**  \n   Localizada próxima ao Mercado Público, essa construção histórica abriga exposições temporárias, eventos culturais e é um espaço dedicado à arte e cultura local.\n\n---\n\nSe quiser, posso também sugerir passeios culturais combinados com as opções gastronômicas que você já tem, para que aproveite ao máximo sua estadia em Florianópolis!"</ul>
}

In [10]:
chain3 = template_cultural | llm |StrOutputParser() # A saída do llm é uma string, por isso, utilizamos o StrOutputParser

chain = chain1 | chain2 | chain3 # A saída da cadeia 1 é a entrada da cadeia 2, e assim por diante. Para utilizar a variável cidade

resposta = chain.invoke({"interesse":"praias"}) # DIFERENTE DO QUE ACONTECEU COM O PROMPT TEMPLATE, AQUI INVOCAMOS A CADEIA, E COMTÉM UM TEMPLATE, E ELA INVOCA A LLM
print(resposta)

Claro! Além da excelente gastronomia que você já conhece em Florianópolis, a cidade tem uma rica cena cultural que vale a pena explorar. Aqui vão algumas sugestões de atividades e locais culturais para aproveitar:

### Atividades Culturais
1. **Visita ao Mercado Público de Florianópolis**  
   Além de ser um ótimo lugar para degustar frutos do mar, o Mercado é um ponto histórico da cidade, com lojas de artesanato, música ao vivo e eventos culturais.

2. **Museu Histórico de Santa Catarina (Palácio Cruz e Sousa)**  
   Localizado no centro, o museu conta a história da região com exposições permanentes e temporárias, além de arquitetura histórica.

3. **Centro Integrado de Cultura (CIC)**  
   Um dos maiores centros culturais da cidade, com teatro, cinema, biblioteca e exposições de arte. Sempre tem programação variada com shows, peças teatrais e exposições.

4. **Projeto Tamar - Centro de Visitantes Florianópolis**  
   Para quem gosta de natureza e educação ambiental, o Projeto Tamar o